# Placement V4 PVE gameplay benchmark

이 Notebook은 같은 7-bag stream을 heuristic과 frozen V4에 공급한 raw Node report를 실행하거나 읽고, paired gameplay 차이를 해석하는 안내서다. 모델 학습이나 production 승격은 수행하지 않는다.


## 1. 실험 질문과 고정 조건

**왜:** imitation metric은 실제 생존을 보장하지 않으므로 authoritative gameplay에서 별도로 확인한다.

**질문:** 동일한 결정적 7-bag stream에서 frozen V4가 기존 heuristic보다 큰 gameplay 회귀 없이 생존하는가?

**고정:** 빈 초기 board, hard execution profile, incoming garbage 없음, 같은 bag, 같은 placement cap, 기존 `processAiStep()`과 authoritative core. 바뀌는 값은 placement policy 하나뿐이다.


## 2. 입력과 frozen artifact 확인

**입력:** frozen model과 Node runner의 project-relative 경로를 사용한다. SHA가 다르면 report를 실행하거나 읽지 않는다.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess

ROOT = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'scripts/run-placement-v4-pve-gameplay-benchmark.js').is_file()
)
MODEL_PATH = ROOT / 'data/models/placement-v4-125g-candidate/model.json'
RUNNER_PATH = ROOT / 'scripts/run-placement-v4-pve-gameplay-benchmark.js'
EXPECTED_MODEL_SHA256 = 'ff64b0d3e4cd9eb6c6d7920f2cbe961f1d70e4c90ccc69a5d9ab8bcd410cb33d'
REPORT_PATH = Path(os.environ.get(
    'PLACEMENT_V4_GAMEPLAY_REPORT',
    '/private/tmp/placement-v4-pve-gameplay-smoke-notebook.json',
))
TIMING_PATH = Path(os.environ.get(
    'PLACEMENT_V4_GAMEPLAY_TIMING_REPORT',
    '/private/tmp/placement-v4-pve-gameplay-smoke-notebook-timing.json',
))

assert MODEL_PATH.is_file(), f'Model missing: {MODEL_PATH.relative_to(ROOT)}'
assert RUNNER_PATH.is_file(), f'Runner missing: {RUNNER_PATH.relative_to(ROOT)}'
model_sha256 = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
assert model_sha256 == EXPECTED_MODEL_SHA256, (model_sha256, EXPECTED_MODEL_SHA256)
{'model': str(MODEL_PATH.relative_to(ROOT)), 'modelSha256': model_sha256}


## 3. Raw report 실행 또는 로딩

`PLACEMENT_V4_GAMEPLAY_RUN=1`이면 Notebook이 Node runner를 실행한다. 그렇지 않으면 이미 생성한 report를 읽는다. Node가 게임 규칙과 rollout을 담당하고 Python은 결과를 재구성하지 않는다.


In [ ]:
run_requested = os.environ.get('PLACEMENT_V4_GAMEPLAY_RUN', '0') == '1'
stream_count = int(os.environ.get('PLACEMENT_V4_GAMEPLAY_STREAMS', '3'))
max_placements = int(os.environ.get('PLACEMENT_V4_GAMEPLAY_MAX_PLACEMENTS', '50'))
seed_prefix = os.environ.get('PLACEMENT_V4_GAMEPLAY_SEED_PREFIX', 'placement-v4-gameplay-smoke')
if run_requested:
    command = [
        'node', str(RUNNER_PATH),
        '--model', str(MODEL_PATH),
        '--output', str(REPORT_PATH),
        '--timing-output', str(TIMING_PATH),
        '--streams', str(stream_count),
        '--max-placements', str(max_placements),
        '--seed-prefix', seed_prefix,
    ]
    subprocess.run(command, cwd=ROOT, check=True)

assert REPORT_PATH.is_file(), f'Run the Node report first: {REPORT_PATH}'
report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
assert report['modelSha256'] == EXPECTED_MODEL_SHA256
assert report['benchmarkSchemaVersion'] == 2
assert report['productionIntegration'] is False
assert report['canonicalArtifact'] is False
assert report['config'] == {
    'executionProfile': 'hard',
    'incomingGarbage': False,
    'maxPlacements': max_placements,
    'policies': ['heuristic', 'placement_v4'],
    'seedPrefix': seed_prefix,
    'streamCount': stream_count,
}, 'stale or different cohort report'
is_fixed_full_cohort = (
    stream_count == 30 and max_placements == 500
    and seed_prefix == 'placement-v4-gameplay-full'
)
assert report['runMode'] == ('full' if is_fixed_full_cohort else 'smoke')
assert report['fullCohortCovered'] is is_fixed_full_cohort
assert report['policyDecisionEligible'] == (
    is_fixed_full_cohort and report['status'] != 'gameplay_benchmark_safety_failed'
)
assert report['canonicalResultEligible'] is False
{'status': report['status'], 'pairs': len(report['pairs']), 'config': report['config']}


## 4. Paired gameplay 비교

각 행은 같은 bag checksum을 공유하는 heuristic/V4 한 쌍이다. cap/cap은 양쪽 모두 horizon 생존, cap/game-over는 cap policy의 survival 우세, game-over/game-over의 같은 placement 수만 exact tie다. `placementDelta`는 보조 수치다.


In [ ]:
paired_rows = []
for pair in report['pairs']:
    paired_rows.append({
        'streamId': pair['streamId'],
        'bagSha': pair['bagStreamSha256'][:12],
        'heuristicPlacements': pair['heuristic']['metrics']['placementCount'],
        'v4Placements': pair['v4']['metrics']['placementCount'],
        'placementDelta': pair['placementDelta'],
        'heuristicLinesPer100': pair['heuristic']['metrics']['linesPer100Placements'],
        'v4LinesPer100': pair['v4']['metrics']['linesPer100Placements'],
    })
paired_rows


## 5. 안전성과 fallback 진단

invalid decision, model fallback, plan divergence, non-finite score, unexpected error는 성능 수치보다 먼저 확인한다. 하나라도 0이 아니면 smoke를 gameplay 근거로 해석하지 않는다.


In [ ]:
SAFETY_KEYS = [
    'invalidDecision', 'modelFallback', 'planDivergence',
    'nonFiniteScore', 'unexpectedError',
]
safety_totals = {key: 0 for key in SAFETY_KEYS}
for pair in report['pairs']:
    for policy_result in [pair['heuristic'], pair['v4']]:
        for key in SAFETY_KEYS:
            safety_totals[key] += policy_result['safety'][key]
assert all(value == 0 for value in safety_totals.values()), safety_totals
{'safety': safety_totals, 'pairedSummary': report['summary']}


## 6. Non-canonical timing 진단

first decision latency와 decision latency p50/p95는 머신 부하에 따라 달라진다. 따라서 timing 파일은 canonical report SHA와 정책 판단 payload에 포함하지 않는다.


In [ ]:
timing = json.loads(TIMING_PATH.read_text(encoding='utf-8')) if TIMING_PATH.is_file() else None
if timing is not None:
    assert timing['canonical'] is False
timing


## 7. Smoke 해석과 사용자 full-run gate

Smoke는 3 paired streams × 2 policies × 최대 50 placements다. 파이프라인과 안전성만 확인하며 gameplay promotion 결론을 내리지 않는다.

Full screening은 사용자 승인 후에만 다음 고정 cohort로 실행한다: `--streams 30 --max-placements 500 --seed-prefix placement-v4-gameplay-full`. 총 60 rollout이며, safety가 0이고 heuristic horizon count와 V4 count 차이가 3 이하면 screening pass다. 이는 production 승격이나 우월 판정이 아니다.

수동 smoke는 별도 `GAME_EVENTS_LOG_PATH`로 heuristic normal, V4 normal, V4 hard를 각각 한 게임 실행한다. V4 model은 `data/models/placement-v4-125g-candidate/model.json`을 사용한다. 서버 오류, ticker 정지, AI placement, garbage와 game-over를 확인하며 V4 성공 log에 heuristic placement가 있으면 fallback으로 판정한다.
